# Synthetic ESM Observation Generator Demo

This notebook demonstrates how to use `esm_observation_generator.py` to create passive-ESM observations that are consistent with the aircraft/radar knowledge graph generated by `kg_generator.py`.

Each observation contains:

- radar-mode fields that are plausibly interceptable or inferable by an ESM sensor;
- measured radar parameters with uncertainty intervals;
- estimated emitter location and an error box;
- approximate kinematics with errors;
- Unix and ISO-8601 UTC timestamps;
- a ground-truth aircraft/operator/radar/mode label and ambiguity candidates.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path
import json

# Run from the repository root. If opened elsewhere, adjust this path.
REPO_ROOT = Path.cwd()
GENERATED_DIR = REPO_ROOT / "generated"
GENERATED_DIR.mkdir(exist_ok=True)

## Generate observations in memory

The Python API is deterministic for a fixed seed. Date bounds are timezone-aware UTC datetimes.

In [ ]:
from esm_observation_generator import generate_observations

observations_doc = generate_observations(
    count=2500,
    seed=42,
    start=datetime(2025, 1, 1, tzinfo=UTC),
    end=datetime(2025, 1, 8, tzinfo=UTC),
)

observations_doc["metadata"]

## Inspect one synthetic observation

The example below shows the top-level structure. The full object includes the ESM radar features, location estimate, kinematics, ambiguity candidates, and the ground-truth label.

In [ ]:
first = observations_doc["observations"][0]
first.keys()

In [ ]:
print(json.dumps(first["ground_truth_label"], indent=2))

In [ ]:
print(json.dumps(first["sensor"], indent=2)[:2000])

In [ ]:
print(json.dumps(first["esm_radar_parameters"], indent=2)[:2000])

## Location and kinematic uncertainty

Emitter location is represented as a point estimate plus a latitude/longitude error box. Kinematics are represented by sampled speed/altitude/heading values and associated error ranges.

In [ ]:
print(json.dumps(first["estimated_emitter_location"], indent=2))
print(json.dumps(first["approximate_kinematics"], indent=2))

## Ambiguity candidates

Some observations intentionally remain compatible with multiple aircraft/variants. The generator includes candidates sharing major KG features, such as radar family or aircraft family, to support probabilistic assignment experiments.

In [ ]:
first["candidate_labels_from_shared_kg_features"]

## Confirm labels resolve to existing KG nodes

This check verifies that each generated ground-truth aircraft, radar, and radar-mode ID exists in the KG.

In [ ]:
from kg_generator import generate_graph

graph = generate_graph()
node_ids = {node["id"] for node in graph["nodes"]}

missing = []
for obs in observations_doc["observations"]:
    label = obs["ground_truth_label"]
    for key in ("aircraft_id", "radar_id", "mode_id"):
        if label[key] not in node_ids:
            missing.append((obs["observation_id"], key, label[key]))

assert not missing, missing
print(f"Validated {len(observations_doc['observations'])} observations against {len(node_ids)} KG nodes.")

## Save observations to JSON

The generated document can be written directly to disk for downstream r-GCN, Dempster-Shafer, or other probabilistic-fusion experiments.

In [ ]:
output_path = GENERATED_DIR / "esm_observations_notebook_demo.json"
output_path.write_text(json.dumps(observations_doc, indent=2, sort_keys=True) + "\n", encoding="utf-8")
output_path

## Equivalent CLI usage

The same workflow is available from the command line:

```bash
python esm_observation_generator.py \
  --count 10 \
  --seed 42 \
  --start 2025-01-01T00:00:00Z \
  --end 2025-01-08T00:00:00Z \
  --output generated/esm_observations_notebook_demo.json
```